In [0]:
print("Hello")

In [0]:
from datetime import datetime
batch_id = f"bronze_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

In [0]:
import gzip, zipfile, io

path = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-02.Z"

data = open(path, "rb").read()
print("First bytes:", data[:2], "| size:", len(data), "bytes")

if data[:2] == b"PK":
    z = zipfile.ZipFile(io.BytesIO(data))
    print("ZIP containing:", z.namelist())
    text = z.read(z.namelist()[0]).decode("latin-1")
else:
    text = gzip.decompress(data).decode("latin-1")

lines = text.splitlines()
print(len(lines), "rows")
for line in lines[:5]:
    print(line)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
mkt_summary_schema = StructType([
    StructField('trade_date',StringType(),True),
    StructField('symbol',StringType(),True),
    StructField('sector_code',StringType(),True),
    StructField('company_name',StringType(),True),
    StructField('open_raw',StringType(),True),
    StructField('high_raw',StringType(),True),
    StructField('low_raw',StringType(),True),
    StructField('close_raw',StringType(),True),
    StructField("volume_raw",StringType(), True),
    StructField("ldcp_raw",StringType(), True),  
    StructField("extra_1",StringType(), True),   
    StructField("extra_2",StringType(), True),   
    StructField("extra_3",StringType(), True)
])

In [0]:
import gzip, zipfile, io, os

src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-02.Z"
out = "/Volumes/workspace/bronze/raw_landing/_extracted/2015/2015-01-02.txt"

data = open(src, "rb").read()
if data[:2] == b"PK":
    z = zipfile.ZipFile(io.BytesIO(data))
    text = z.read(z.namelist()[0]).decode("latin-1")
else:
    text = gzip.decompress(data).decode("latin-1")

os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    f.write(text)

print("Wrote", len(text.splitlines()), "lines to", out)

In [0]:
df = (spark.read
      .schema(mkt_summary_schema)
      .option("sep", "|")
      .option("header", "false")
      .csv(out))

df.printSchema()
print("Rows:", df.count())
display(df.limit(10))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.mkt_summary (
  trade_date       STRING,
  symbol           STRING,
  sector_code      STRING,
  company_name     STRING,
  open_raw         STRING,
  high_raw         STRING,
  low_raw          STRING,
  close_raw        STRING,
  volume_raw       STRING,
  ldcp_raw         STRING,
  extra_1          STRING,
  extra_2          STRING,
  extra_3          STRING,
  file_trade_date  DATE,
  source_file      STRING,
  batch_id         STRING,
  load_timestamp   TIMESTAMP
)
USING DELTA
PARTITIONED BY (file_trade_date)

In [0]:
from pyspark.sql import functions as F


df_bronze = (df
    .withColumn("file_trade_date", F.lit("2015-01-02").cast("date"))
    .withColumn("source_file",     F.col("_metadata.file_path"))
    .withColumn("batch_id",        F.lit("test_run_1"))
    .withColumn("load_timestamp",  F.current_timestamp())
)

df_bronze = df_bronze.select(
    "trade_date", "symbol", "sector_code", "company_name",
    "open_raw", "high_raw", "low_raw", "close_raw", "volume_raw", "ldcp_raw",
    "extra_1", "extra_2", "extra_3",
    "file_trade_date", "source_file", "batch_id", "load_timestamp"
)

display(df_bronze.limit(5))

In [0]:
# Step 2: write to the Bronze table
df_bronze.write.mode("append").saveAsTable("workspace.bronze.mkt_summary")
print("Written:", df_bronze.count(), "rows")

In [0]:
%sql

SELECT COUNT(*) AS rows, MIN(load_timestamp), MAX(load_timestamp)
FROM workspace.bronze.mkt_summary;

SELECT * FROM workspace.bronze.mkt_summary LIMIT 10;

In [0]:
import gzip, zipfile, io, os
from pyspark.sql import functions as F

def load_to_bronze(date, batch_id, load_type):
  print("Loading", date, "with batch_id", batch_id, "and load_type", load_type)
  year = date[:4]
  src = "/Volumes/workspace/bronze/raw_landing/mkt_summary/" + year + "/" + date + ".Z"
  out = "/Volumes/workspace/bronze/raw_landing/_extracted/" + year + "/" + date + ".txt"

  # 1. Decompress .Z (ZIP or GZIP) to a staging .txt file
  data = open(src, "rb").read()
  if data[:2] == b"PK":
      z = zipfile.ZipFile(io.BytesIO(data))
      text = z.read(z.namelist()[0]).decode("latin-1")
  else:
      text = gzip.decompress(data).decode("latin-1")

  os.makedirs(os.path.dirname(out), exist_ok=True)
  with open(out, "w", encoding="utf-8") as f:
      f.write(text)

  print("Wrote", len(text.splitlines()), "lines to", out)

  # 2. Read with the explicit schema
  df = (spark.read
        .schema(mkt_summary_schema)
        .option("sep", "|")
        .option("header", "false")
        .csv(out))

  # 3. Add label columns
  df_bronze = (df
    .withColumn("file_trade_date", F.lit(date).cast("date"))
    .withColumn("source_file",     F.lit(src))
    .withColumn("batch_id",        F.lit(batch_id))
    .withColumn("load_timestamp",  F.current_timestamp())
  )

  df_bronze = df_bronze.select(
    "trade_date", "symbol", "sector_code", "company_name",
    "open_raw", "high_raw", "low_raw", "close_raw", "volume_raw", "ldcp_raw",
    "extra_1", "extra_2", "extra_3",
    "file_trade_date", "source_file", "batch_id", "load_timestamp"
  )

  # 4. Write to Bronze
  df_bronze.write.mode("append").saveAsTable("workspace.bronze.mkt_summary")

  # 5. Return number of rows written
  row_count = df_bronze.count()
  print("Loaded", row_count, "rows for", date)
  return row_count

In [0]:
rows = load_to_bronze("2015-01-05", "test", "backfill")